# CIA contest experiment

Covers the paper Tables 1–3 experiment (not Table 9).

Run from the repository root.

In [15]:
%cd /content
# https://github.com/shaz01/metricdp-flower-pytorch branch: new_experiments
![ -d metricdp-pytorch ] || git clone -b new_experiments https://github.com/shaz01/metricdp-flower-pytorch metricdp-pytorch
%cd metricdp-pytorch

# --no-deps keeps Colab's preinstalled CUDA-matched torch/torchvision intact
%pip install -e . --no-deps
%pip install "flwr[simulation]>=1.32.1,<2.0" "datasets>=3.0,<5.0" "numpy>=2.0" "scikit-learn>=1.9.0"

/content
/content/metricdp-pytorch
Obtaining file:///content/metricdp-pytorch
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for metricdp-pytorch (pyproject.toml) ... done
  Created wheel for metricdp-pytorch: filename=metricdp_pytorch-0.1.0-py3-none-any.whl size=9379 sha256=6f3962867d7f9efa553127a7027afe7cc73f62a8fc0d8150b3ad45c174d55d17
  Stored in directory: /tmp/pip-ephem-wheel-cache-r941qc30/wheels/ac/68/b2/5a03af700a7a0c97a74cfd72af7962d83bf647533b4b0b3564
Successfully built metricdp-pytorch
  Attempting uninstall: metricdp-pytorch
    Found existing installation: metricdp-pytorch 0.1.0
    Uninstalling metricdp-pytorch-0.1.0:
      Successfully uninstalled metricdp-pytorch-0.1.0


In [14]:
import subprocess, sys
from dataclasses import replace
from experiments.cia.scripts.contest import MATRIX, NUM_CLIENTS, OUTPUT_DIR

matrix_1 = replace(MATRIX, seeds=(42,))
combo = matrix_1.list_combos(name_prefix="contest", num_clients=NUM_CLIENTS)[0]
command = [
  sys.executable, "-m", "experiments.reproduce.runner",
  *combo.runner_args(
      output_dir=OUTPUT_DIR,
      max_parallel_clients=4,
      client_cpus=1.0,
      checkpoint_rounds=(1, 20),
  ),
]
print(command)
result = subprocess.run(command, capture_output=True, text=True)
print("returncode:", result.returncode)
print("--- stdout ---")
print(result.stdout)
print("--- stderr ---")
print(result.stderr)

['/usr/bin/python3', '-m', 'experiments.reproduce.runner', '--data-module', 'experiments.cia.scripts.contest:create_data_module', '--model-module', 'experiments.reproduce.paper_cnn:create_model', '--num-clients', '4', '--partition', 'homogeneous', '--privacy', 'vanilla', '--aggregation', 'fedavg', '--seed', '42', '--noise-multiplier', '0.01', '--clipping-norm', '5.0', '--rounds', '20', '--local-epochs', '5', '--batch-size', '32', '--learning-rate', '0.001', '--initialization-epochs', '20', '--max-parallel-clients', '4', '--client-cpus', '1.0', '--output-dir', '/content/metricdp-pytorch/results/reproduce', '--run-name', 'contest__homogeneous__vanilla__fedavg__clients-4__seed-42__nm0p01__clip5__rounds-20__epochs-5__contest__paper_cnn', '--checkpoint-rounds', '1', '20']
returncode: 1
--- stdout ---
{
  "partition-mode": "homogeneous",
  "partition-profile": "auto",
  "client-weights": "",
  "data-module": "experiments.cia.scripts.contest:create_data_module",
  "model-module": "experiments

In [8]:
from collections.abc import Iterable, Mapping
from dataclasses import replace
from pathlib import Path
from typing import Any

from experiments.cia.attack_runner import run_attack
from experiments.cia.datasets.partitions import PartitionViewDataModule, in_remove
from experiments.cia.shadow_dataset import clean_shadow_dataset, noisy_shadow_dataset
from experiments.reproduce.dataset.alzheimer import (
    create_data_module as create_alzheimer_data_module,
)
from experiments.reproduce.matrix import Hyperparams, Matrix
from metricdp_pytorch.utils.device import resolve_device

PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "results" / "reproduce"

NUM_CLIENTS = 4
TARGET_PARTITION_ID = 0
SHADOW_FRACTION = 0.10
NOISE_STD_FRACTION = 0.10

ROUNDS = 20
CHECKPOINT_ROUNDS = (1, ROUNDS)


def create_data_module(config: Mapping[str, Any]) -> PartitionViewDataModule:
    """Build this script's IN-remove participant view over Alzheimer MRI."""
    canonical_num_partitions = int(config["num-clients"])
    return in_remove(
        create_alzheimer_data_module(config),
        canonical_num_partitions=canonical_num_partitions,
        target_partition_id=TARGET_PARTITION_ID,
    )

In [9]:
MATRIX = Matrix(
    partitions=("homogeneous",),
    privacy_modes=("vanilla", "global-dp", "metric-privacy"),
    aggregations=("fedavg",),
    seeds=(),
    noise_multipliers=(0.01,),
    hyperparams=Hyperparams(
        clipping_norm=5.0,
        rounds=ROUNDS,
        local_epochs=5,
        batch_size=32,
        learning_rate=0.001,
        initialization_epochs=20,
    ),
    data_module="experiments.cia.scripts.contest:create_data_module",
    model_module="experiments.reproduce.paper_cnn:create_model",
)


def copy_matrix_with_seeds(matrix: Matrix, seeds: Iterable[int]) -> Matrix:
    """Return a matrix with only its seed dimension replaced."""
    return replace(matrix, seeds=tuple(seeds))


CLEAN_SHADOW_DATASET = lambda combo: clean_shadow_dataset(
    combo,
    target_partition_id=TARGET_PARTITION_ID,
    shadow_fraction=SHADOW_FRACTION,
)

NOISY_SHADOW_DATASET = lambda combo: noisy_shadow_dataset(
    combo,
    target_partition_id=TARGET_PARTITION_ID,
    shadow_fraction=SHADOW_FRACTION,
    std_fraction=NOISE_STD_FRACTION,
)

In [10]:
def run_contest(matrix: Matrix = MATRIX) -> list[Any]:
    combos = matrix.list_combos(name_prefix="contest", num_clients=NUM_CLIENTS)
    results = run_attack(
        combos=combos,
        output_dir=OUTPUT_DIR,
        log_path=OUTPUT_DIR / "progress.log",
        max_parallel_clients=4,
        force=False,
        start_message=f"Reproduction starting: {len(combos)} combinations",
        clean_data_module_factory=CLEAN_SHADOW_DATASET,
        noisy_data_module_factory=NOISY_SHADOW_DATASET,
        device=resolve_device(),
        checkpoint_rounds=CHECKPOINT_ROUNDS,
        report_name="contest.json",
    )
    for result in results:
        print(
            f"round={result.server_round:2d} {result.partition_mode:12s} "
            f"{result.privacy:15s} {result.aggregation:8s} "
            f"agg={result.aggregated_test_loss:.3f} "
            f"clean_target={result.target_clean_shadow_loss:.3f} "
            f"noisy_target={result.target_noisy_shadow_loss:.3f} "
            f"shadow_n={result.shadow_size} "
            f"clean_diff={result.clean_difference_pct:.3f}% "
            f"noisy_diff={result.noisy_difference_pct:.3f}%"
        )
    return results

In [13]:
# Run 1
matrix_1 = replace(MATRIX, seeds=(42,))
results_1_original = run_contest(matrix_1)

# Move run 1's artifacts out of the output directory so the reproducibility
# rerun is neither skipped nor allowed to overwrite the original files.
run_1_archive = OUTPUT_DIR / "contest_run_1_original"
run_1_archive.mkdir(parents=True, exist_ok=False)
for combo in matrix_1.list_combos(name_prefix="contest", num_clients=NUM_CLIENTS):
    result_path = combo.result_path(OUTPUT_DIR)
    result_path.rename(run_1_archive / result_path.name)
for filename in ("contest.json", "progress.log"):
    path = OUTPUT_DIR / filename
    path.rename(run_1_archive / filename)

# Re-run run 1 to check reproducibility and compare with the original results.
results_1_rerun = run_contest(matrix_1)

2026-08-04 13:08:58 Reproduction starting: 3 combinations
2026-08-04 13:08:58 START contest__homogeneous__vanilla__fedavg__clients-4__seed-42__nm0p01__clip5__rounds-20__epochs-5__contest__paper_cnn 
2026-08-04 13:08:58 FAILED contest__homogeneous__vanilla__fedavg__clients-4__seed-42__nm0p01__clip5__rounds-20__epochs-5__contest__paper_cnn (exit=1, 0.4s)
2026-08-04 13:08:58 PROGRESS 1/3 (1 failed so far)
2026-08-04 13:08:58 START contest__homogeneous__global-dp__fedavg__clients-4__seed-42__nm0p01__clip5__rounds-20__epochs-5__contest__paper_cnn 
2026-08-04 13:08:59 FAILED contest__homogeneous__global-dp__fedavg__clients-4__seed-42__nm0p01__clip5__rounds-20__epochs-5__contest__paper_cnn (exit=1, 0.4s)
2026-08-04 13:08:59 PROGRESS 2/3 (2 failed so far)
2026-08-04 13:08:59 START contest__homogeneous__metric-privacy__fedavg__clients-4__seed-42__nm0p01__clip5__rounds-20__epochs-5__contest__paper_cnn 
2026-08-04 13:08:59 FAILED contest__homogeneous__metric-privacy__fedavg__clients-4__seed-42__n

FileExistsError: [Errno 17] File exists: '/content/metricdp-pytorch/results/reproduce/contest_run_1_original'

In [ ]:
# Run 2
matrix_2 = replace(MATRIX, seeds=(43,))
results_2 = run_contest(matrix_2)

In [ ]:
# Run 3
matrix_3 = replace(MATRIX, seeds=(44,))
results_3 = run_contest(matrix_3)